##Requirements

In [2]:
!pip install pymupdf langchain_community langchain transformers sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.3/409.3 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: SQLAlchemy
    Found existing installation: SQLAlchemy 2.0.36
    Uninstalling SQLAlchemy-2.0.36:
      Successfully uninstalled SQLAlchemy-2.0.36
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.15
    Uninstalling langchain-core-0.3.15:
      Successfully uninstalled langchain-core-0.3.15


In [3]:
import tempfile
import os
import threading
from queue import Queue
from concurrent.futures import ThreadPoolExecutor
from typing import List, Tuple, Dict, Set
import fitz
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain.chains import LLMChain, RefineDocumentsChain
from langchain.prompts import PromptTemplate
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.llms import HuggingFacePipeline
from langchain.schema import Document
import spacy
import re
from collections import defaultdict

#1. part-1 : loadig the models and preparing the Refining function:

In [ ]:
# Available models
T5_MODEL = "t5-large"
TRANSLATION_MODEL = "Helsinki-NLP/opus-mt-en-fr"

#setting Default numbers of the output tokens
MAX_SUMMARY_LENGTH_DEFAULT = 200
MIN_SUMMARY_LENGTH_DEFAULT = 30

#Initialize and return a T5 model with output length controls
def load_t5_model(model_name: str, max_length: int = MAX_SUMMARY_LENGTH_DEFAULT, min_length: int = MIN_SUMMARY_LENGTH_DEFAULT):

    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    pipe = pipeline(
        "summarization",
        model=model,
        tokenizer=tokenizer,
        min_length=min_length,
        max_length=max_length,
        num_beams=4,
        early_stopping=True
    )

    return HuggingFacePipeline(pipeline=pipe)


def load_translation_model():
    #Initialize and return the translation model

    tokenizer = AutoTokenizer.from_pretrained(TRANSLATION_MODEL)
    model = AutoModelForSeq2SeqLM.from_pretrained(TRANSLATION_MODEL)

    translation_pipeline = pipeline(
        "translation",
        model=model,
        tokenizer=tokenizer
    )
    return HuggingFacePipeline(pipeline=translation_pipeline)

def create_chain(llm):
    #Create and return the document refinement chain
    document_prompt = PromptTemplate(
        input_variables=["page_content"],
        template="{page_content}"
    )

    summarize_prompt = PromptTemplate(
        input_variables=["context"],
        template="Generate a concise and coherent summary of the following context: {context}"
    )

    initial_llm_chain = LLMChain(llm=llm, prompt=summarize_prompt)

    refine_template = """
    Produce a final summary.

    Existing summary up to this point:
    {existing_answer}

    New context:
    -----
    {context}
    -----

    Given the new context, refine the original summary.
    """

    refine_prompt = PromptTemplate(
        input_variables=["existing_answer", "context"],
        template=refine_template
    )

    refine_llm_chain = LLMChain(llm=llm, prompt=refine_prompt)

    return RefineDocumentsChain(
        initial_llm_chain=initial_llm_chain,
        refine_llm_chain=refine_llm_chain,
        document_prompt=document_prompt,
        document_variable_name="context",
        initial_response_name="existing_answer"
    )

#2. part-2 : porcessing the document :

PDFSection class represents a section of a PDF document

In [ ]:
class PDFSection:
    def __init__(self, title: str, content: str, order: int, level: int = 0):
        self.title = title
        self.content = content
        self.order = order
        self.level = level
        self.summary = ""
        self.translation = ""

In [ ]:
# Load spaCy model at the start - add this near your other model definitions
try:
    nlp = spacy.load("en_core_web_lg")
except OSError:
    os.system("python -m spacy download en_core_web_lg")
    nlp = spacy.load("en_core_web_lg")


**detect_sections** function analyzes PDF structure to identify distinct sections, it's contain :
* **is_likely_title** function that determines if text is likely a section title
* **extract_hierarchical_sections** that extracts sections while maintaining document hierarchy
* **merge_short_sections** to combines short sections with previous sections




In [ ]:
def detect_sections(pdf_path: str) -> List[PDFSection]:

    # Custom title patterns
    TITLE_PATTERNS = [
        r"^(?:Chapter|Section|\d+\.)\s+.*$",  # Matches with "Chapter X", "Section X", "1. Title"
        r"^(?:[A-Z][A-Za-z\s]{2,}:?\s*$)",    # Matches capitalized phrases
        r"^(?:Introduction|Conclusion|Methodology|Hypotheses|Abstract|References|Bibliography|Acknowledgements|Background|Case Study|Discussion)$",  # Common section names
    ]
    title_regex = re.compile('|'.join(TITLE_PATTERNS), re.MULTILINE)

    #Determine if text is likely a title using multiple heuristics
    def is_likely_title(text: str, font_size: float, is_bold: bool, prev_font_size: float) -> bool:

        if not text.strip():
            return False

        # Process text with spaCy
        doc = nlp(text)

        # Linguistic features that suggest a title
        has_subject = any(token.dep_ == "nsubj" for token in doc)
        is_nominal = doc[0].pos_ in {"NOUN", "PROPN"}
        word_count = len(text.split())

        # Combine multiple signals
        signals = [
            title_regex.match(text) is not None,  # Matches title patterns
            is_bold and font_size > prev_font_size ,  # Visual emphasis and Size difference from context
            is_bold and font_size > 11,  # Size difference from context
            is_nominal,  # Starts with a noun
            has_subject,  # Has a subject
            word_count <= 10,  # Not too long
            text[0].isupper() if text else False,  # Starts with capital letter
        ]

        # Weight the signals
        signal_weights = [4, 3, 2, 1.5, 1, 2, 1] # Adjustable weights
        score = sum(weight * signal for weight, signal in zip(signal_weights, signals))

        return score >= 9

     #Extract sections while maintaining hierarchical structure.
    def extract_hierarchical_sections(doc: fitz.Document) -> List[PDFSection]:

        sections = []
        current_section = None
        current_content = []
        section_order = 0
        prev_font_size = 12  # Default font size
        font_size_levels = defaultdict(int)  # Track font sizes for hierarchy

        for page in doc:
            blocks = page.get_text("dict")["blocks"]

            for block in blocks:
                if "lines" not in block:
                    continue

                for line in block["lines"]:
                    for span in line["spans"]:
                        text = span["text"].strip()
                        font_size = span["size"]
                        is_bold = span["flags"] & 2**4 != 0

                        if is_likely_title(text, font_size, is_bold, prev_font_size):
                            # Determine section level based on font size
                            if font_size not in font_size_levels:
                                font_size_levels[font_size] = len(font_size_levels)
                            level = font_size_levels[font_size]

                            # Save previous section if exists
                            if current_section:
                                sections.append(PDFSection(
                                    current_section,
                                    "\n".join(current_content),
                                    section_order,
                                    level
                                ))
                                section_order += 1

                            current_section = text
                            current_content = []
                        else:
                            current_content.append(text)

                        prev_font_size = font_size

        # Add the last section
        if current_section:
            level = font_size_levels[prev_font_size]
            sections.append(PDFSection(
                current_section,
                "\n".join(current_content),
                section_order,
                level
            ))

        return sections

    #Merge sections that are too short (under 50 words) with their previous section.
    def merge_short_sections(sections: List[PDFSection]) -> List[PDFSection]:

        merged_sections = []
        for i, section in enumerate(sections):
            if i > 0 and len(section.content.split()) < 50:
                merged_sections[-1].content += f"\n\n{section.title}\n{section.content}"
            else:
                merged_sections.append(section)
        return merged_sections

    # Process the PDF
    doc = fitz.open(pdf_path)
    try:
        sections = extract_hierarchical_sections(doc)

        # Post-process sections to clean and validate
        for section in sections:
            # Clean up title and content
            section.title = section.title.strip()
            section.content = re.sub(r'\s+', ' ', section.content).strip()

            # Analyze content with spaCy for validation
            if len(section.content) > 100:  # Only analyze substantial sections
                content_doc = nlp(section.content[:1000])  # Analyze first 1000 chars
                # Validate section has meaningful content
                has_sentences = any(sent.root.pos_ == "VERB" for sent in content_doc.sents)
                if not has_sentences:
                    continue

        sections = merge_short_sections(sections)
        # Return only non-empty sections
        return [s for s in sections if s.content.strip()]

    finally:
        doc.close()

The purpose of process_section is to be used in the **process_pdf** function to process individual extracted sections and pass them into the summary chain and the translation model.





In [ ]:
#Process a single section with summarization and translation
def process_section(section: PDFSection, chain, translation_model, results_queue: Queue, max_length: int = MAX_SUMMARY_LENGTH_DEFAULT):

    try:
       #the chunk size & chunk_overlap dynamicly change based on the max_output_length
        chunk_size = round(1500 *(max_length / 200))
        chunk_overlap = round(chunk_size * 0.06)

        # Split text into chunks
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )
        chunks = text_splitter.split_text(section.content)
        documents = [Document(page_content=chunk) for chunk in chunks]

        # Generate summary
        summary_result = chain.invoke({"input_documents": documents})
        section.summary = summary_result['output_text']

        # Translate summary
        translation_result = translation_model.invoke(section.summary)
        section.translation = translation_result

        # Put the processed section in the queue
        results_queue.put((section.order, section))

    except Exception as e:
        print(f"Error processing section '{section.title}': {str(e)}")
        results_queue.put((section.order, None))


In [ ]:
#Process all the PDF file with parallel section processing
def process_pdf(pdf_file : str , chain, translation_model, max_length: int = MAX_SUMMARY_LENGTH_DEFAULT):

    tmp_path = pdf_file
    try:
        # Detect sections
        sections = detect_sections( tmp_path)

        if not sections:
           sections = [PDFSection("Main Content", PyMuPDFLoader(tmp_path).load()[0].page_content, 0)]

        # Process sections in parallel
        results_queue = Queue()
        threads = []

        with ThreadPoolExecutor(max_workers=len(sections)) as executor:
            # Submit each section for processing
            futures = []
            for section in sections:
                # Submit the task and store the future
                future = executor.submit(
                    process_section,  # Function to execute
                    section=section,
                    chain=chain,
                    translation_model=translation_model,
                    results_queue=results_queue,
                    max_length=max_length
                )
                futures.append(future)

            # Wait for all futures to complete
            for future in futures:
                future.result()

        # Collect results in order
        processed_sections = []
        for _ in range(len(sections)):
            order, section = results_queue.get()
            if section:
                processed_sections.append((order, section))

        # Sort sections by original order
        processed_sections.sort(key=lambda x: x[0])

        return [section for _, section in processed_sections]

    finally:
        os.unlink(tmp_path)


###2.1 testing the fuction of detecting sections:

the pdf example from : https://arxiv.org/pdf/2411.04242

In [ ]:
test=detect_sections("/content/drive/MyDrive/Book/Multimodal-Structure-Aware-Quantum-Data-Processing.pdf")

In [ ]:
i=0
for i in range (len(test)):
 print(test[i].title)
 print(test[i].content)

Multimodal Structure-Aware Quantum Data Processing


Hala Hawashin
University College London hala.hawashin.23@ucl.ac.uk

Mehrnoosh Sadrzadeh
University College London m.sadrzadeh@ucl.ac.uk
Abstract
While large language models (LLMs) have ad- vanced the field of natural language processing (NLP), their “black box” nature obscures their decision-making processes. To address this, researchers developed structured approaches using higher order tensors. These are able to model linguistic relations, but stall when training on classical computers due to their ex- cessive size. Tensors are natural inhabitants of quantum systems and training on quantum computers provides a solution by translating text to variational quantum circuits. In this paper, we develop MultiQ -NLP: a framework for structure-aware data processing with multi- modal text+image data. Here, “structure” refers to syntactic and grammatical relationships in language, as well as the hierarchical organiza- tion of visual elements 

#3. testing the implementation on the pdf

In [ ]:
model = load_t5_model(T5_MODEL)
translation_model = load_translation_model()
chain = create_chain(model)

In [ ]:
pdf_file = "/content/drive/MyDrive/Book/Multimodal-Structure-Aware-Quantum-Data-Processing.pdf"

In [ ]:
summery_test=process_pdf(pdf_file, chain, translation_model)

In [ ]:
for section in summery_test:
  print("\n",section.title,"\n",section.translation)


 Multimodal Structure-Aware Quantum Data Processing 
 Hala hawashin, ucl, london. Mehrnoosh sadrzadeh, uk.

 Abstract 
 Les grands modèles de langage (LLM) ont progressé dans le domaine du traitement du langage naturel (NLP) mais leur nature « black box » obscurcit leurs processus décisionnels. Dans cet article, nous développons MultiQ -NLP : un cadre pour le traitement des données structurales avec des données text+image multi-modales.

 Quantum computing is an emerging technology 
 Lambeq est un outil de programmation de haut niveau pour le traitement quantique du langage naturel (qnlp) il peut générer un résumé concis et cohérent du contexte suivant . il est le premier cadre pour le traitement de données structurauxware qui intègre le langage et d'autres sources de données, comme les images .

 Related Work 
 Les circuits quantiques sont utilisés pour modéliser la structure de composition du langage . approche s'harmonise avec des modèles catégoriques dans NLP qui considèrent la gr